<a href="https://colab.research.google.com/github/nKhaled2578/FlyRank-ML-Internship-Nourhan/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nKhaled2578/FlyRank-ML-Internship-Nourhan/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Loading the data directly from your public GitHub repo to guarantee it works without HuggingFace errors
print("Loading reliable local dataset from GitHub...")
url = "https://raw.githubusercontent.com/nKhaled2578/FlyRank-ML-Internship-Nourhan/main/data/raw/content_refresh_anonymized.csv"
df_working = pd.read_csv(url)

print(f"✅ Data loaded successfully! Total rows ready for the assignment: {len(df_working)}")




Loading reliable local dataset from GitHub...
✅ Data loaded successfully! Total rows ready for the assignment: 30000


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis (Grain): One row represents a single web page (content_id) and its performance metrics for a specific client over the observed historical window.**

**Time Window: I am using the historical window provided in the dataset to build features, deliberately avoiding looking into future outcomes to prevent data leakage.**

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verifying the dataset is loaded and checking its shape
print(f"Dataset shape verified: {df_working.shape}")

Dataset shape verified: (30000, 44)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

#Features (My 5 choices):
**search_volume: Knowable at the decision moment because it aggregates historical market demand.**  

**cpc: Knowable at the decision moment from historical ad bids.**  

**word_count: Knowable at the decision moment because the content is already written and measured.**  

**avg_position: Knowable at the decision moment as a summary of the past ranking.**  

**ctr: Knowable at the decision moment directly from historical Search Console logs.**

#Label/Proxy:
 **Since my task is Unsupervised Clustering, my output is a Cluster ID. To demonstrate the leakage trap, I will use trend_direction as a binary proxy label.**  

#Context:
 **content_id and client_id.**

#Excluded:
  **trend_pct (Trend Percentage). Excluded because it is mathematically derived from the future outcome data. Including it causes a leakage trap.**  

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
context_cols = ['client_id', 'content_id']
feature_cols = ['search_volume', 'cpc', 'word_count', 'avg_position', 'ctr']
proxy_label = 'trend_direction'
excluded_leak = 'trend_pct'

print(f"Features ready for the model: {feature_cols}")
print(f"Excluded trap feature: {excluded_leak}")

Features ready for the model: ['search_volume', 'cpc', 'word_count', 'avg_position', 'ctr']
Excluded trap feature: trend_pct


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**The queries below prove the grain uniqueness, row count, availability filter (IS TRUE), and finally, spring the Leakage Trap where future data makes the score suspiciously perfect.**  

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Prove the Grain
is_unique = df_working.duplicated(subset=['client_id', 'content_id']).sum() == 0
print(f"Fact 1 - Grain verified (Unique URL per client): {is_unique}")

# 2. Row count
row_count = len(df_working)
print(f"Fact 2 - Row count: {row_count}")

# 3. Availability Filter (IS TRUE)
# Filter to only include active pages (where search volume is strictly greater than 0)
is_active_condition = df_working['search_volume'] > 0
df_active = df_working[is_active_condition == True].copy()
print(f"Fact 3 - Rows surviving availability filter (IS TRUE): {len(df_active)}")

# ==========================================
# 4. THE TRAP: The Leakage Lesson
# ==========================================
# We create a binary proxy label: 1 if the trend is 'down', 0 otherwise
y_proxy = (df_active['trend_direction'] == 'down').astype(int)

# LEAKED MODEL: We sneak in 'trend_pct', which correlates perfectly with the future label
X_leaked = df_active[['search_volume', 'cpc', 'trend_pct']].fillna(0)
model_leaked = LogisticRegression(max_iter=1000)
model_leaked.fit(X_leaked, y_proxy)
score_leaked = accuracy_score(y_proxy, model_leaked.predict(X_leaked))
print(f"\n🚨 THE TRAP (Leaked Score): {score_leaked:.4f} (Suspiciously perfect! We memorized the future.)")

# HONEST MODEL: We remove the leaked column
X_honest = df_active[['search_volume', 'cpc', 'word_count']].fillna(0)
model_honest = LogisticRegression(max_iter=1000)
model_honest.fit(X_honest, y_proxy)
score_honest = accuracy_score(y_proxy, model_honest.predict(X_honest))
print(f"✅ HONEST Score: {score_honest:.4f} (The real, baseline performance)")

Fact 1 - Grain verified (Unique URL per client): True
Fact 2 - Row count: 30000
Fact 3 - Rows surviving availability filter (IS TRUE): 16451

🚨 THE TRAP (Leaked Score): 0.9999 (Suspiciously perfect! We memorized the future.)
✅ HONEST Score: 0.5576 (The real, baseline performance)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Limitation: This data slice suffers from an unbalanced history. It only covers established pages with active search volume. It completely ignores brand-new pages, meaning our model will have a blind spot for new content until it gathers enough historical GSC data.**

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
dropped_pages = len(df_working) - len(df_active)
print(f"Limitation impact: {dropped_pages} inactive pages were excluded.")

Limitation impact: 13549 inactive pages were excluded.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.